In [1]:
import os
import json
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np
from sklearn import metrics
from sklearn.metrics import brier_score_loss
import torch
from torchmetrics.classification import BinaryCalibrationError
ece_metric = BinaryCalibrationError(n_bins=10, norm='l1')
nll_metric = torch.nn.BCELoss()

In [2]:
dataset = targetset = "Virus" #"Virus" "Tumor" "Bacteria"

EPSILON = 1e-12

In [3]:
PLM_LIST = ['esmc_600m', 'Rostlab/ProstT5', 'ElnaggarLab/ankh-large', 'facebook/esm2_t33_650M_UR50D', 'Rostlab/prot_bert']

In [4]:
PLM_dict = {
    'esmc_600m': 'ESMC',
    'Rostlab/ProstT5': 'ProstT5',
    'ElnaggarLab/ankh-large': 'Ankh',
    'facebook/esm2_t33_650M_UR50D': 'ESM2',
    'Rostlab/prot_bert': 'Prot Bert'
}

In [5]:
filepath = f'./{dataset}-{targetset}.pkl'

import pickle
with open(filepath, 'rb') as file:
    result_dict = pickle.load(file)

In [6]:
# result_dict

In [7]:
def kl_divergence_normal(mu1, sigma1, mu2, sigma2, epsilon:float=1e-12):
# def kl_divergence_normal(mu2, sigma2, mu1, sigma1):
    """
    Calculates the KL divergence between two univariate normal distributions.

    Args:
        mu1 (float): Mean of the first normal distribution (P).
        sigma1_sq (float): Variance of the first normal distribution (P).
        mu2 (float): Mean of the second normal distribution (Q).
        sigma2_sq (float): Variance of the second normal distribution (Q).

    Returns:
        float: The KL divergence KL(P||Q).
    """

    kl_div = 0.5 * (np.log(sigma2**2 / (epsilon + sigma1**2)) + 
                    (sigma1**2 + (mu1 - mu2)**2) / sigma2**2 - 1)
    return kl_div

def CDiv(y_true, y_pred_mean, y_pred_std, ref_mean, ref_std):
    # y_pred_mean = y_pred_mc.mean(axis=-1)
    # y_pred_std = y_pred_mc.std(axis=-1)
    unc_mean = ref_mean * np.ones(y_pred_mean.shape)
    unc_std = ref_std * np.ones(y_pred_std.shape)
    kl_score = torch.tensor(kl_divergence_normal(y_pred_mean, y_pred_std, unc_mean, unc_std))
    y_true = torch.tensor(2*y_true - 1).unsqueeze(-1)
    score = torch.log(torch.tensor(y_pred_mean)/(1 + EPSILON- torch.tensor(y_pred_mean)))
    score *= y_true

    return (score * kl_score).numpy(), score.numpy(), kl_score.numpy()

In [8]:
def get_ensemble_results_same_plm(result_dict, weighting_strategy, plm_model_name:str, method:str, c:float=5.0):

    pred_prob, pred_unct, labl = [], [], []
    for protein_name in result_dict:
        pred_data = result_dict[protein_name][plm_model_name]
        pred_p, pred_u = [], []
        for key in pred_data[method]:
            pred_p.append(pred_data[method][key]['prob'])
            pred_u.append(pred_data[method][key]['unct'])
        pred_prob.append(pred_p)
        pred_unct.append(pred_u)
        labl.append(pred_data[method][key]['labl'])

    y_pred = np.array(pred_prob)
    y_unct = np.array(pred_unct)
    y_true = np.array(labl)

    y_pred_prob = y_pred

    if weighting_strategy == "kld":
        ref_mean, ref_std = 0.5, 0.1
        unc_mean = ref_mean * np.ones(y_pred.shape)
        unc_std = ref_std * np.ones(y_unct.shape)
        weight = kl_divergence_normal(y_pred, y_unct, unc_mean, unc_std)
        norm_weight = weight/np.sum(weight, axis=-1, keepdims=True)
        y_pred = np.sum(norm_weight * y_pred, axis=-1)
    elif weighting_strategy == "unbiased":
        weight = 1.0/(y_unct**2+EPSILON)
        norm_weight = weight/np.sum(weight, axis=-1, keepdims=True)
        y_pred = np.sum(norm_weight * y_pred, axis=-1)
    elif weighting_strategy == "ns":
        weight = np.exp(-c*y_unct)
        norm_weight = weight/np.sum(weight, axis=-1, keepdims=True)
        y_pred = np.sum(norm_weight * y_pred, axis=-1)

    cm_matrix = metrics.confusion_matrix(np.array(y_pred >= 0.5, dtype=np.int8), y_true)

    precision = cm_matrix[1][1] / max(EPSILON, cm_matrix[1][1]+cm_matrix[0][1])
    recall = cm_matrix[1][1] / max(EPSILON, cm_matrix[1][1]+cm_matrix[1][0])
    f1_score = (2*precision*recall) / max(EPSILON, precision+recall)
    accuracy = (cm_matrix[0][0]+cm_matrix[1][1]) / (cm_matrix[0][0]+cm_matrix[1][1]+cm_matrix[1][0]+cm_matrix[0][1])

    fpr, tpr, thresholds = metrics.roc_curve(np.array(y_true)+1, y_pred, pos_label=2)
    auc_roc = metrics.auc(fpr, tpr)

    bsl = brier_score_loss(y_true, y_pred)

    ece = ece_metric(torch.tensor(y_pred, dtype=torch.float), torch.tensor(y_true, dtype=torch.int)).item()
    nll = nll_metric(torch.tensor(y_pred, dtype=torch.float), torch.tensor(y_true, dtype=torch.float)).item()

    return accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl, y_pred_prob, y_true, y_unct

In [9]:
def get_ensemble_results_same_seed(result_dict, weighting_strategy, seed:int, method:str, c:float=5.0):

    pred_prob, pred_unct, labl = [], [], []
    for protein_name in result_dict:
        pred_p, pred_u = [], []
        for plm_model_name in result_dict[protein_name]:
            pred_data = result_dict[protein_name][plm_model_name]
            pred_p.append(pred_data[method][seed]['prob'])
            pred_u.append(pred_data[method][seed]['unct'])
        pred_prob.append(pred_p)
        pred_unct.append(pred_u)
        labl.append(pred_data[method][seed]['labl'])

    y_pred = np.array(pred_prob)
    y_unct = np.array(pred_unct)
    y_true = np.array(labl)

    y_pred_prob = y_pred

    if weighting_strategy == "kld":
        ref_mean, ref_std = 0.5, 0.1
        unc_mean = ref_mean * np.ones(y_pred.shape)
        unc_std = ref_std * np.ones(y_unct.shape)
        weight = kl_divergence_normal(y_pred, y_unct, unc_mean, unc_std)
        norm_weight = weight/np.sum(weight, axis=-1, keepdims=True)
        y_pred = np.sum(norm_weight * y_pred, axis=-1)
    elif weighting_strategy == "unbiased":
        weight = 1.0/(y_unct**2+EPSILON)
        norm_weight = weight/np.sum(weight, axis=-1, keepdims=True)
        y_pred = np.sum(norm_weight * y_pred, axis=-1)
    elif weighting_strategy == "ns":
        weight = np.exp(-c*y_unct)
        norm_weight = weight/np.sum(weight, axis=-1, keepdims=True)
        y_pred = np.sum(norm_weight * y_pred, axis=-1)

    cm_matrix = metrics.confusion_matrix(np.array(y_pred >= 0.5, dtype=np.int8), y_true)

    precision = cm_matrix[1][1] / max(EPSILON, cm_matrix[1][1]+cm_matrix[0][1])
    recall = cm_matrix[1][1] / max(EPSILON, cm_matrix[1][1]+cm_matrix[1][0])
    f1_score = (2*precision*recall) / max(EPSILON, precision+recall)
    accuracy = (cm_matrix[0][0]+cm_matrix[1][1]) / (cm_matrix[0][0]+cm_matrix[1][1]+cm_matrix[1][0]+cm_matrix[0][1])

    fpr, tpr, thresholds = metrics.roc_curve(np.array(y_true)+1, y_pred, pos_label=2)
    auc_roc = metrics.auc(fpr, tpr)

    bsl = brier_score_loss(y_true, y_pred)

    ece = ece_metric(torch.tensor(y_pred, dtype=torch.float), torch.tensor(y_true, dtype=torch.int)).item()
    nll = nll_metric(torch.tensor(y_pred, dtype=torch.float), torch.tensor(y_true, dtype=torch.float)).item()

    return accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl, y_pred_prob, y_true, y_unct

WE can create an ensemble in two ways. 
- One way is use the same PLM for all UQ models and train with different experimental seeds.
- Another way is to use the same experimental seeds but different PLM for all UQ models.

# DUNE with same plm but different seeds

In [10]:
c = 5.0
weighting_strategy = 'unbiased' #'kld' #'unbiased' #'ns'

plm_model_name = 'ESMC' #'ESMC' #'ProstT5' #'Ankh' #'ESM2' #'Prot Bert'
method = 'DROPOUT' #'SVDKL' #'SVDKL32' #'SVDKL128'#'LA' #'LA32' #'LA128' #'TS' #'DVBLL' #'DVBLL32' #'DVBLL128' #'SGLD' #'SWAG' #'DROPOUT'

accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl, y_pred, y_true, y_unct = get_ensemble_results_same_plm(result_dict, weighting_strategy, plm_model_name, method, c)
cdiv, *_ = CDiv(y_true, y_pred, y_unct, ref_mean=0.5, ref_std=0.1)

In [11]:
accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl, cdiv.mean(axis=-1).mean()

(0.9269521410579346,
 0.9478908188585607,
 0.9116945107398569,
 0.9294403892944039,
 0.9761761215436655,
 0.02971278689801693,
 0.2134643793106079,
 0.0583694833521437,
 52.23802971149259)

# DUNE with same seed but different plms

In [12]:
c = 5.0
weighting_strategy = 'unbiased' #'kld' #'unbiased' #'ns'

seed = 1 #1 #2 #3 #4 #5
method = 'DROPOUT' #'SVDKL' #'SVDKL32' #'SVDKL128'#'LA' #'LA32' #'LA128' #'TS' #'DVBLL' #'DVBLL32' #'DVBLL128' #'SGLD' #'SWAG' #'DROPOUT'

accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl, y_pred, y_true, y_unct = get_ensemble_results_same_seed(result_dict, weighting_strategy, seed, method, c)
cdiv, *_ = CDiv(y_true, y_pred, y_unct, ref_mean=0.5, ref_std=0.1)

In [13]:
accuracy, precision, recall, f1_score, auc_roc, ece, nll, bsl, cdiv.mean(axis=-1).mean()

(0.9193954659949622,
 0.9429280397022333,
 0.9026128266033254,
 0.9223300970873788,
 0.9727110609051042,
 0.034709382802248,
 0.21862569451332092,
 0.060003799926124514,
 46.624154822569416)